In [2]:
#Importando df bronze
from radar_b3.pipeline import ingerir_dados
from radar_b3.spark_session import criar_spark_session

df_bronze = ingerir_dados()

spark = criar_spark_session()
df_bronze_spark = spark.createDataFrame(df_bronze)

df_bronze_spark.show(10, truncate=False)

c:\Users\lipet\Documents\radar-b3\.venv\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
c:\Users\lipet\Documents\radar-b3\.venv\Lib\site-packages\yfinance\utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
c:\Users\lipet\Documents\radar-b3\.venv\Lib\site-packages\yfinance\utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
c:\Users\lipet\Documents\radar-b3\

+-------------------+------+----------+-----+------------+--------+--------+--------+--------+--------+
|Date               |ticker|tipo_ativo|setor|benchmark_id|Close   |High    |Low     |Open    |Volume  |
+-------------------+------+----------+-----+------------+--------+--------+--------+--------+--------+
|2026-04-06 00:00:00|^BVSP |benchmark |NULL |ibovespa    |188162.0|189220.0|187811.0|188054.0|5697600 |
|2026-04-07 00:00:00|^BVSP |benchmark |NULL |ibovespa    |188259.0|188259.0|185885.0|188162.0|8753200 |
|2026-04-08 00:00:00|^BVSP |benchmark |NULL |ibovespa    |192201.0|193759.0|188260.0|188261.0|12594000|
|2026-04-09 00:00:00|^BVSP |benchmark |NULL |ibovespa    |195129.0|195514.0|192206.0|192206.0|9382900 |
|2026-04-10 00:00:00|^BVSP |benchmark |NULL |ibovespa    |197324.0|197554.0|195129.0|195130.0|9661000 |
|2026-04-13 00:00:00|^BVSP |benchmark |NULL |ibovespa    |198001.0|198173.0|196223.0|197324.0|9947000 |
|2026-04-14 00:00:00|^BVSP |benchmark |NULL |ibovespa    |198657

In [4]:
from pyspark.sql.functions import col, sum

df_bronze_spark.select(
    sum(col("Open").isNull().cast("int")).alias("Open_nulos"),
    sum(col("High").isNull().cast("int")).alias("High_nulos"),
    sum(col("Low").isNull().cast("int")).alias("Low_nulos"),
    sum(col("Close").isNull().cast("int")).alias("Close_nulos")
).show()

+----------+----------+---------+-----------+
|Open_nulos|High_nulos|Low_nulos|Close_nulos|
+----------+----------+---------+-----------+
|         0|         0|        0|          0|
+----------+----------+---------+-----------+



In [ ]:
#Teste de duplicidade
df_duplicatas = df_bronze_spark.groupBy("Date","ticker").count().filter("count > 1")
df_duplicatas.show()

+----+------+-----+
|Date|ticker|count|
+----+------+-----+
+----+------+-----+



In [22]:
#Regra de Negócio LOW-HIGH
df_errohl = df_bronze_spark.filter(df_bronze_spark.High < df_bronze_spark.Low)
df_errohl.show()

#Erro Close fora de High e Low
df_errocl = df_bronze_spark.filter((df_bronze_spark.Close > df_bronze_spark.High) | (df_bronze_spark.Close < df_bronze_spark.Low))
df_errocl.show()

+----+------+----------+-----+------------+-----+----+---+----+------+
|Date|ticker|tipo_ativo|setor|benchmark_id|Close|High|Low|Open|Volume|
+----+------+----------+-----+------------+-----+----+---+----+------+
+----+------+----------+-----+------------+-----+----+---+----+------+

+----+------+----------+-----+------------+-----+----+---+----+------+
|Date|ticker|tipo_ativo|setor|benchmark_id|Close|High|Low|Open|Volume|
+----+------+----------+-----+------------+-----+----+---+----+------+
+----+------+----------+-----+------------+-----+----+---+----+------+



In [45]:
import radar_b3.transformations.prices as prices
import importlib

importlib.reload(prices)

problemas = prices.validar_qualidade_precos(df_bronze_spark)
print(problemas)

[]


In [ ]:
#Testar erro forçado de close
from pyspark.sql.functions import when, col, lit

df_teste = df_bronze_spark.withColumn(
    "Close",
    when(col("ticker") == "ITUB4.SA", lit(-100.0))
    .otherwise(col("Close"))
)

problemas_teste = prices.validar_qualidade_precos(df_teste)

print(problemas_teste)

['Erro na qualidade dos dados. Close']
